parsing pdf

In [ ]:
1
from pypdf import PdfReader

import os

file_path = "../data/pdf/intern.pdf"

# print(os.path.exists(file_path))
# print(os.path.getsize(file_path))

reader = PdfReader(file_path)

text = ""

for page in reader.pages:
    text += page.extract_text()
# print(text[:1000])

def chunk_text(text, chunk_size=100):

    words = text.split()

    chunks = []

    for i in range(0, len(words), chunk_size):

        chunk = " ".join(words[i:i+chunk_size])
        # print(chunk)

        chunks.append(chunk)

    return chunks
    
chunks = chunk_text(text)

# print(chunks[0])
# print(len(chunks))

Take-Home Assignment: Python Developer Intern Role: Python Developer Intern Time allowed: 3 days from receipt Submission: Google Drive folder link shared to maria@sustainablelivinglab.org Note: This is a fictional scenario designed to assess your skills. The system described below does not exist, is not in use, and will not be used by SL2 or any other organisation. We are evaluating how you think and build, not collecting a deliverable. Before You Start Don't spend more than 2-3 days on this. That is the intended scope. We only want to see what you can build in that window, not a polished product
you've spent a week on. Send us your submission even if it's incomplete. A partial API that works is far more useful to us than nothing. What's the worst that can happen? You lose 2-3 days and we give you honest feedback. Don't give up because something broke - document it and submit anyway. We are not just looking for Python developers. We are looking for AI-first builders. This means we expe

Embanding

In [ ]:
# LOAD EMBEDDING MODEL
from sentence_transformers import SentenceTransformer

# Load model once
model = SentenceTransformer("all-MiniLM-L6-v2")

d:\AllProjects\FastApi\projects_anti\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\AllProjects\FastApi\projects_anti\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an a

In [20]:
# create embanding
def create_embedding(text: str):

    embedding = model.encode(text)

    return embedding.tolist()

In [21]:
# create chunk to embanding
def create_embeddings_for_chunks(chunks):

    vectors = []

    for chunk in chunks:

        vector = create_embedding(chunk)

        vectors.append({
            "text": chunk,
            "embedding": vector
        })

    return vectors

In [22]:
# test ist
vector = create_embedding("low stock items in pune")

print(len(vector))
print(vector[:5])

384
[0.061466705054044724, -0.018917860463261604, -0.018823500722646713, -0.020528411492705345, 0.002537403255701065]


ADDING CHROMODB

In [23]:
import chromadb

# Persistent DB storage
client = chromadb.PersistentClient(path="../data/db/chroma_db")

# Create collection
collection = client.get_or_create_collection(
    name="documents"
)

In [24]:
# STORE CHUNKS + EMBEDDINGS
def store_chunks(vectors):

    for idx, item in enumerate(vectors):

        collection.add(
            ids=[str(idx)],
            documents=[item["text"]],
            embeddings=[item["embedding"]]
        )

In [25]:
# SIMILARITY SEARCH
def search_similar_chunks(query_embedding, top_k=3):

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    return results["documents"][0]

In [31]:
# TEST FULL FLOW
chunks = chunk_text(text)

vectors = create_embeddings_for_chunks(chunks)

store_chunks(vectors)

query_vector = create_embedding("Task 1: Data Model and Core API")

results = search_similar_chunks(query_vector)

print(results)

Take-Home Assignment: Python Developer Intern Role: Python Developer Intern Time allowed: 3 days from receipt Submission: Google Drive folder link shared to maria@sustainablelivinglab.org Note: This is a fictional scenario designed to assess your skills. The system described below does not exist, is not in use, and will not be used by SL2 or any other organisation. We are evaluating how you think and build, not collecting a deliverable. Before You Start Don't spend more than 2-3 days on this. That is the intended scope. We only want to see what you can build in that window, not a polished product
you've spent a week on. Send us your submission even if it's incomplete. A partial API that works is far more useful to us than nothing. What's the worst that can happen? You lose 2-3 days and we give you honest feedback. Don't give up because something broke - document it and submit anyway. We are not just looking for Python developers. We are looking for AI-first builders. This means we expe

In [33]:
# create llm
import requests

OLLAMA_URL = "http://localhost:11434/api/generate"

def ask_llm(prompt: str):

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": "llama3.2:3b",
            "prompt": prompt,
            "stream": False
        }
    )

    return response.json()["response"]

In [37]:
# 
def ask_rag(question: str):

    # STEP 1 → Create question embedding
    query_embedding = create_embedding(question)

    # STEP 2 → Retrieve similar chunks
    chunks = search_similar_chunks(query_embedding)

    # STEP 3 → Merge chunks into context
    context = "\n".join(chunks)

    # STEP 4 → Build prompt
    prompt = f"""
You are an AI assistant.

Answer ONLY from the provided context.

Context:
{context}

Question:
{question}
"""

    # STEP 5 → Ask LLM
    answer = ask_llm(prompt)

    return {
        "question": question,
        "context": chunks,
        "answer": answer
    }

result = ask_rag("Data Model and Core API?")

print(result)

{'question': 'Data Model and Core API?', 'context': ["are free to use what you know. If you're starting fresh, these all have free tiers: Layer Recommended Option Framework FastAPI (preferred) or Flask Database Neon (PostgreSQL) Deployment Railway / Render / Fly.io Auth tokens PyJWT or python-jose If you use a different service or library for any layer, document your choice in the README. Tasks Task 1: Data Model and Core API Design and implement a REST API with the following schema and endpoints. Entities: • users: id, name, email, hashed_password, role (student / trainer / institution / programme_manager / monitoring_officer), institution_id (nullable), created_at • batches: id, name, institution_id,", 'attendance records, so the summary endpoints return meaningful data. Task 2: JWT Authentication and Role-Based Access Control Implement authentication across the full API. Signup and login: • POST /auth/signup: accepts name, email, password, and role; stores a hashed password (use bcr